# Notebook 20 — Parameter-set wiring audit (Day 16)

## Purpose

Day 15 closed three X6 isolation phases (composite NE, physical-init,
OCP hysteresis) with deferred or reframed status. Each was blocked by
a wiring or feasibility issue specific to the Chen2020 parameter family
in PyBaMM 26.3.1. Day 16 advances the project beyond Chen2020 by
auditing which alternative parameter sets are runnable and along which
axes they support chemistry-transferability testing or mechanism-
specific isolation tests that were impossible inside Chen2020.

The notebook scope is **parameter-family transferability audit**, not 
mechanism-isolation batch. The latter follows once a viable parameter 
set is locked.

## Day 16 first stage — 4-axis parameter-set audit

Every candidate parameter set is evaluated along four axes:

(a) **Runnable on PyBaMM 26.3.1** — builds and solves a 1-second rest 
    on a DFN model with `thermal: lumped`
(b) **Supports DCAC protocol** — accepts `CustomStepExplicit` with 
    `I(t) = -|I_DC| + |I_AC|·sin(2πft)`, reaches the chemistry-specific 
    upper voltage cut-off via `direction='charge'`. Cutoff voltage is 
    read from `pv["Upper voltage cut-off [V]"]`, not hard-coded.
(c) **Natively supports hysteresis / composite / strong-OCP-plateau** — 
    parameter set provides keys for non-default OCP options, has 
    intrinsic OCP plateau structure (LFP), or provides composite 
    electrode formulation
(d) **Does not trigger V_min infeasibility** — V_init headroom above 
    chemistry-specific lower voltage cut-off allows DCAC sin discharge 
    half-cycle without immediate boundary violation in high-AC anchors

### Eligibility logic (revised)

A parameter set is **chemistry-transfer batch eligible** if axes (a), 
(b), and (d) pass. Axis (c) is **not** a general exclusion criterion; 
it determines which mechanism-specific tests are natively supported by 
that parameter family. A set without hysteresis keys may still be valid 
for chemistry-transfer testing, but not for a clean OCP-hysteresis 
isolation test.

A set failing axis (a) is wiring-fix follow-up. Failing (b) or (d) 
excludes from chemistry-transfer batch.

### Day 16 selection order (explicit lock)

1. **First choice**: NMC / NCA-like parameter set (same chemistry 
   family as Chen2020). Transferability isolated to a single axis 
   (parameter values within the same chemistry).
2. **Fallback**: LFP / strong-plateau parameter set. Used only if no 
   NMC/NCA-like set passes axes (a)+(b)+(d).
3. **If both available**: NMC/NCA first as primary batch, LFP as 
   contrast follow-up. This avoids confounding chemistry shift with 
   parameter set internals on the first attempt.

## Methodology lock (per Day 14-15 cumulative findings)

### Δt(Q) integration convention
Strict-net accounting: Q_net(t) integrated from signed I(t), no 
rectification, no monotonic enforcement. Negative-current intervals 
retained. Target times are first-passage times on Q_net(t).

### Metric layer
- **A_Δt** = primary scalar metric (per-case)
- **Full Δt(Q) curve** = mandatory audit layer alongside A_Δt
- **trajectory_class** (discrete threshold) deprecated as primary 
  verdict observable per Day 14 #0 robustness audit

### V_min audit (mandatory after Day 15, chemistry-agnostic)

Every batch must include columns:

- `V_min_charge` (instantaneous min during charging)
- `Vmin_cutoff_V` (parameter-set's lower voltage cut-off)
- `V_below_Vmin_fraction` (fraction of charging time below cut-off)
- `low_voltage_class` ∈ {clean, transient_low_voltage, 
  severe_low_voltage, infeasible}

**Mechanism evidence gate** (chemistry-specific tolerance):
- For Chen2020-lineage comparisons, transient cases with 
  `V_below_Vmin_fraction < 0.02` are admitted with a boundary note, 
  following the Day 13/14 empirical audit
- For new parameter families, the 0.02 value is reported as a reference 
  threshold and must be re-evaluated after the first feasibility audit. 
  LFP / NCA / other chemistries may require different empirical 
  tolerances based on their baseline V_min behaviour.

`severe_low_voltage` cases are recorded but not admitted as mechanism 
evidence regardless of chemistry.

### DCAC waveform convention (Day 14 #1 lock)

- Canonical: `I(t) = -|I_DC| + |I_AC|·sin(2πft)`
- PyBaMM convention: I > 0 = discharge, I < 0 = charge
- First sin half-cycle (sin ≥ 0) → discharge → V dip expected
- Frequency labeling preserved from Chen2020 protocol; 1τ = 11.1s 
  reference. New parameter sets may have different time constants — 
  protocol frequency labels are normalized via Chen2020-derived f_Hz, 
  not re-derived per parameter set, to maintain comparability of 
  Δt(Q) anchors.

### Ablation comparison rule (Day 12 lock)

A_Δt window is fixed per case across baseline + all ablations. 
Self-adaptive window per ablation is forbidden. For chemistry-transfer 
testing, the per-case window is derived from each parameter family's 
baseline (not Chen2020's window applied across families).

### Writing discipline (Day 12-13 lock)

- Negative findings: "NOT SUPPORTED within tested range" with 
  quantified scope. Never EXCLUDED / RULED OUT / INSUFFICIENT.
- Positive findings: "supported as candidate" until magnitude grid + 
  repeat audit + window integrity + trajectory check
- External docs (notebook intro, commit, README, paper) never 
  reference internal memory indices

### PyBaMM 26.3.1 quirks (Day 15 lock)

- Avoid `sim.build()` followed by `sim.solve()` on the same Simulation 
  object in this experiment probe workflow (raises 
  `'symbol_processor' ValueError`). Use direct `sim.solve()`.
- Each experiment string → its own cycle: `sol.cycles[i].steps[0]`
- "Hold V until I" requires Ampere notation ('0.0368 A'), not mA strings
- V_min event emits as warning, not exception; check V_end / Q_CC_end / 
  t_charge to verify feasibility

### Cell parameterization caveat

Chen2020 is not the measured MJ1 cell parameterization. Sign mismatch 
between simulation and MJ1 experiment is not automatically a model 
failure — it is a generalizability test output (per Memory #12 frame).

## Phase 1 candidate parameter sets (hypotheses only)

The names below are hypotheses; Cell 1 derives the authoritative list 
from `pybamm.parameter_sets`. Some names below may not exist in this 
PyBaMM 26.3.1 distribution; Cell 1 will reveal which.

NMC / NCA-family hypotheses:
- Chen2020 (control / reference, already verified)
- Chen2020_composite (X6α blocker; included for completeness)
- Marquis2019
- OKane2022
- ORegan2022
- NCA_Kim2011 (NCA hypothesis)
- Mohtat2020
- Ai2020

LFP / strong-OCP-plateau hypotheses:
- Prada2013
- Wycisk2022
- Ramadass2004 (older NMC, unclear if still in distribution)
- Ecker2015 (older NMC)

Cell 1 enumerates `pybamm.parameter_sets`, classifies each by chemistry 
tag (where metadata available), then probes axis (a) on each.

In [1]:
# ============================================================
# Cell 1 (rev) — Day 16 parameter-set discovery + axis (a) probe
# 
# axis (a) split into three independent probes, each with FRESH 
# ParameterValues to avoid cross-probe state contamination:
#   A1  DFN  + native default     + 1s rest
#   A2  DFN  + set_initial_state(0.05) + 1s rest
#   A3  SPMe + native default     + 1s rest      (diagnostic only)
# 
# Eligibility logic NOT applied here. Cell 1 only records the four
# axes (a/d/c/metadata) and ships them to CSV. Batch admission to
# Cells 2-4 is decided after stdout review.
# ============================================================

import pybamm
import numpy as np
import pandas as pd
import warnings
from pathlib import Path
import time as _time

print("=== Cell 1 (rev) — Day 16 axis (a) probe, decoupled A1/A2/A3 ===")
print(f"PyBaMM version: {pybamm.__version__}\n")

# === 1. Enumerate ===
params_list = sorted(list(pybamm.parameter_sets))
print(f"Found {len(params_list)} registered parameter sets:")
for p in params_list:
    print(f"  - {p}")
print()


def _fresh_pv(name):
    """Return a brand-new ParameterValues. Setting ambient/init temp here
    is harmless across chemistries; later cells can override per-batch."""
    pv = pybamm.ParameterValues(name)
    pv["Ambient temperature [K]"] = 293.15
    pv["Initial temperature [K]"] = 293.15
    return pv


def _probe(pset_name, model_class, do_set_soc, soc_value=0.05):
    """Run a single probe with a fresh pv. Returns (status, V_init, error)."""
    try:
        pv = _fresh_pv(pset_name)
        if do_set_soc:
            pv.set_initial_state(soc_value)
        model = model_class(options={"thermal": "lumped"})
        exp = pybamm.Experiment(["Rest for 1 second"])
        sim = pybamm.Simulation(model, parameter_values=pv, experiment=exp)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            sol = sim.solve()
        v_arr = sol["Voltage [V]"].entries
        if len(v_arr) >= 1 and np.isfinite(v_arr[0]):
            return "ok", float(v_arr[0]), ""
        return "fail", None, "empty or non-finite voltage array"
    except Exception as e:
        return "fail", None, f"{type(e).__name__}: {str(e)[:200]}"


# === 2. Run three probes per parameter set ===
print("=" * 72)
print("Probing A1 (DFN native) / A2 (DFN set_soc=0.05) / A3 (SPMe native)")
print("=" * 72)

records = []
for name in params_list:
    print(f"\n[{name}]")
    rec = {"param_set": name}

    # Static metadata (one fresh pv, read-only)
    try:
        pv0 = _fresh_pv(name)
        rec["V_min_spec"] = (
            float(pv0["Lower voltage cut-off [V]"])
            if "Lower voltage cut-off [V]" in pv0.keys() else None
        )
        rec["V_max_spec"] = (
            float(pv0["Upper voltage cut-off [V]"])
            if "Upper voltage cut-off [V]" in pv0.keys() else None
        )
        rec["nominal_capacity_Ah"] = (
            float(pv0["Nominal cell capacity [A.h]"])
            if "Nominal cell capacity [A.h]" in pv0.keys() else None
        )
        rec["metadata_load_error"] = ""
    except Exception as e:
        rec["V_min_spec"] = None
        rec["V_max_spec"] = None
        rec["nominal_capacity_Ah"] = None
        rec["metadata_load_error"] = f"{type(e).__name__}: {str(e)[:150]}"

    # Chemistry hint via official docstring API
    try:
        doc = pybamm.parameter_sets.get_docstring(name) or ""
        first_para = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
        rec["docstring_excerpt"] = first_para[:200]
    except Exception as e:
        rec["docstring_excerpt"] = f"(docstring fetch failed: {type(e).__name__})"

    # A1: DFN native default
    t0 = _time.time()
    s, v, err = _probe(name, pybamm.lithium_ion.DFN, do_set_soc=False)
    rec["A1_DFN_native_status"] = s
    rec["A1_DFN_native_V_init"] = v
    rec["A1_DFN_native_error"] = err
    rec["A1_time_s"] = round(_time.time() - t0, 2)

    # A2: DFN + set_initial_state(0.05)
    t0 = _time.time()
    s, v, err = _probe(name, pybamm.lithium_ion.DFN, do_set_soc=True, soc_value=0.05)
    rec["A2_DFN_set_soc_0p05_status"] = s
    rec["A2_DFN_set_soc_0p05_V_init"] = v
    rec["A2_DFN_set_soc_0p05_error"] = err
    rec["A2_time_s"] = round(_time.time() - t0, 2)

    # A3: SPMe native default (DIAGNOSTIC ONLY — does not gate batch admission)
    t0 = _time.time()
    s, v, err = _probe(name, pybamm.lithium_ion.SPMe, do_set_soc=False)
    rec["A3_SPMe_native_status"] = s
    rec["A3_SPMe_native_V_init"] = v
    rec["A3_SPMe_native_error"] = err
    rec["A3_time_s"] = round(_time.time() - t0, 2)

    # Per-set summary line
    a1_v = (f"{rec['A1_DFN_native_V_init']:.4f}V"
            if rec['A1_DFN_native_V_init'] is not None else "-")
    a2_v = (f"{rec['A2_DFN_set_soc_0p05_V_init']:.4f}V"
            if rec['A2_DFN_set_soc_0p05_V_init'] is not None else "-")
    a3_v = (f"{rec['A3_SPMe_native_V_init']:.4f}V"
            if rec['A3_SPMe_native_V_init'] is not None else "-")
    vmin_str = (f"{rec['V_min_spec']:.2f}"
                if rec['V_min_spec'] is not None else "-")
    vmax_str = (f"{rec['V_max_spec']:.2f}"
                if rec['V_max_spec'] is not None else "-")
    cap_str = (f"{rec['nominal_capacity_Ah']:.2f}"
               if rec['nominal_capacity_Ah'] is not None else "-")
    print(f"  V cutoff [{vmin_str}, {vmax_str}] V, Q_nom={cap_str} Ah")
    print(f"  A1 DFN native   : {rec['A1_DFN_native_status']:>4s}  V_init={a1_v}")
    print(f"  A2 DFN set_soc  : {rec['A2_DFN_set_soc_0p05_status']:>4s}  V_init={a2_v}")
    print(f"  A3 SPMe native  : {rec['A3_SPMe_native_status']:>4s}  V_init={a3_v}  [diagnostic]")
    if rec['A1_DFN_native_status'] == "fail":
        print(f"  A1 err: {rec['A1_DFN_native_error']}")
    if rec['A2_DFN_set_soc_0p05_status'] == "fail":
        print(f"  A2 err: {rec['A2_DFN_set_soc_0p05_error']}")
    if rec['A3_SPMe_native_status'] == "fail":
        print(f"  A3 err: {rec['A3_SPMe_native_error']}")

    records.append(rec)

# === 3. Summary table ===
df = pd.DataFrame(records)

print("\n" + "=" * 72)
print("Summary — A1 is the primary axis (a) verdict; A2/A3 are diagnostic")
print("=" * 72)
header = (f"  {'param_set':<22s} | {'A1':>3s} {'A2':>3s} {'A3':>3s} |"
          f" {'V_min':>5s} {'V_max':>5s} {'Q_Ah':>5s}")
print(header)
print("  " + "-" * (len(header) - 2))
for _, r in df.iterrows():
    a1 = "ok" if r['A1_DFN_native_status'] == "ok" else "x"
    a2 = "ok" if r['A2_DFN_set_soc_0p05_status'] == "ok" else "x"
    a3 = "ok" if r['A3_SPMe_native_status'] == "ok" else "x"
    vmn = f"{r['V_min_spec']:.2f}" if r['V_min_spec'] is not None else "-"
    vmx = f"{r['V_max_spec']:.2f}" if r['V_max_spec'] is not None else "-"
    cap = f"{r['nominal_capacity_Ah']:.2f}" if r['nominal_capacity_Ah'] is not None else "-"
    print(f"  {r['param_set']:<22s} | {a1:>3s} {a2:>3s} {a3:>3s} |"
          f" {vmn:>5s} {vmx:>5s} {cap:>5s}")

# === 4. Save ===
out = Path("/Users/louislu/pybamm-dcac-superimposed") / "data" / "day16_step1_axis_a_audit.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print(f"\n[wrote] {out}")
print(f"\nNo eligibility verdict issued. Review stdout, then decide Cell 2 candidates.")

=== Cell 1 (rev) — Day 16 axis (a) probe, decoupled A1/A2/A3 ===
PyBaMM version: 26.3.1

Found 18 registered parameter sets:
  - Ai2020
  - Chayambuka2022
  - Chen2020
  - Chen2020_composite
  - ECM_Example
  - Ecker2015
  - Ecker2015_graphite_halfcell
  - MSMR_Example
  - Marquis2019
  - Mohtat2020
  - NCA_Kim2011
  - OKane2022
  - OKane2022_graphite_SiOx_halfcell
  - ORegan2022
  - Prada2013
  - Ramadass2004
  - Sulzer2019
  - Xu2019

Probing A1 (DFN native) / A2 (DFN set_soc=0.05) / A3 (SPMe native)

[Ai2020]
  V cutoff [3.00, 4.20] V, Q_nom=2.28 Ah
  A1 DFN native   :   ok  V_init=4.1844V
  A2 DFN set_soc  :   ok  V_init=3.5456V
  A3 SPMe native  :   ok  V_init=4.1844V  [diagnostic]

[Chayambuka2022]
  V cutoff [2.00, 4.20] V, Q_nom=0.00 Ah
  A1 DFN native   : fail  V_init=-
  A2 DFN set_soc  : fail  V_init=-
  A3 SPMe native  : fail  V_init=-  [diagnostic]
  A1 err: KeyError: "Parameter 'Negative current collector thickness [m]' not found. 'Negative current collector thickness [m]

In [2]:
# Cell 1.5 — chemistry tag inspection from saved CSV
# Read docstring_excerpt for the 8 axis-(a)-passing sets, plus probe
# active material names (more reliable than docstring keyword matching).

import pybamm
import pandas as pd
from pathlib import Path

CSV = Path("/Users/louislu/pybamm-dcac-superimposed/data/day16_step1_axis_a_audit.csv")
df = pd.read_csv(CSV)

passed = df[df["A1_DFN_native_status"] == "ok"].copy()
print(f"Inspecting {len(passed)} axis-(a)-passing sets:\n")

# Active material name keys (these are strings in pv, more reliable than docstring)
mat_keys = [
    "Negative electrode active material",
    "Positive electrode active material",
]

rows = []
for name in passed["param_set"]:
    pv = pybamm.ParameterValues(name)
    row = {"param_set": name}
    for k in mat_keys:
        row[k] = pv.get(k, "(not set)") if k in pv.keys() else "(no key)"
    # Docstring first 300 chars
    try:
        ds = pybamm.parameter_sets.get_docstring(name) or ""
        row["docstring_head"] = ds.strip().replace("\n", " ")[:300]
    except Exception as e:
        row["docstring_head"] = f"(err: {type(e).__name__})"
    rows.append(row)

out = pd.DataFrame(rows)

# Print per-set
for _, r in out.iterrows():
    print(f"[{r['param_set']}]")
    print(f"  NE active material: {r['Negative electrode active material']}")
    print(f"  PE active material: {r['Positive electrode active material']}")
    print(f"  docstring: {r['docstring_head']}")
    print()

Inspecting 8 axis-(a)-passing sets:

[Ai2020]
  NE active material: (no key)
  PE active material: (no key)
  docstring: Parameters for the Enertech cell (Ai2020), from the papers :footcite:t:`Ai2019`, :footcite:t:`Rieger2016` and references therein.  SEI parameters are example parameters for SEI growth from the papers :footcite:t:`Ramadass2004`, :footcite:t:`Ploehn2004`, :footcite:t:`Single2018`, :footcite:t:`Safari2

[Chen2020]
  NE active material: (no key)
  PE active material: (no key)
  docstring: Parameters for an LG M50 cell, from the paper :footcite:t:`Chen2020` and references therein.  SEI parameters are example parameters for SEI growth from the papers :footcite:t:`Ramadass2004`, :footcite:t:`Ploehn2004`, :footcite:t:`Single2018`, :footcite:t:`Safari2008`, and :footcite:t:`Yang2017`  .. 

[Ecker2015]
  NE active material: (no key)
  PE active material: (no key)
  docstring: Parameters for a Kokam SLPB 75106100 cell, from the papers :footcite:t:`Ecker2015i` and :footcite:t:`E

## Cell 2 — Day 16 protocol feasibility

**Note on Cell 2 rev1**: The first attempt disabled model voltage events 
on the assumption that solving over the full horizon would enable cleaner 
post-hoc V_min auditing. This caused all 48 simulations to fail: without 
V_max event termination, DFN solver runs beyond the physical domain 
(c_s_p → 0 at full charge) and crashes numerically. A minimum viable test 
on Chen2020 confirmed that keeping model events resolves the issue 
(8454 solver steps, terminated naturally at V_max, with V_min audit 
available on the returned trajectory).

**Corrected path**:
- `pv["Current function [A]"] = callable` using `pybamm.sin` (symbolic 
  compatibility with solver autodiff)
- `model.events` untouched — V_max / V_min terminate the simulation 
  naturally at physical-domain boundaries
- Verdict reads `sol.termination` directly:

| `sol.termination`               | + V_min audit         | verdict |
|---------------------------------|-----------------------|---------|
| `event: Maximum voltage [V]`    | clean                 | `feasible_clean` |
| `event: Maximum voltage [V]`    | transient_low_voltage | `feasible_transient` |
| `event: Maximum voltage [V]`    | severe_low_voltage    | `boundary_confounded` |
| `event: Minimum voltage [V]`    | any                   | `infeasible_Vmin_event` |
| `final time` / no voltage event | any                   | `infeasible_no_Vmax` |
| solver exception                | —                     | `solve_error` |

Trajectories that terminate at V_min are not extrapolated to "what would 
have happened after". Cell 3 computes Δt(Q) only on `feasible_clean` 
and `feasible_transient` pairs.

Currents: `I_DC = -|DC_C|·Q_nom`, `I_AC = |AC_C|·Q_nom`. Frequency labels 
preserve MJ1/Chen2020 convention: τ_label = 11.1 s, `f(nτ) = 1/(2π·n·τ_label)`.

In [3]:
# ============================================================
# Cell 2 (rev) — Day 16 protocol feasibility + V_min boundary screen
# 
# Strategy:
#   (1) Remove voltage events from model only (not concentration events)
#   (2) Solve over full horizon without voltage termination
#   (3) Post-hoc detect first Vmax crossing → truncate trajectory there
#   (4) Audit V_min on the [0, first_Vmax_idx] slice only
#   (5) Verdict logic: 
#         not reached Vmax           → infeasible_no_Vmax
#         clean V trajectory         → feasible_clean
#         transient_low_voltage      → feasible_transient
#         severe_low_voltage         → boundary_confounded
#       (severe is "boundary_confounded": Vmax reached but Vmin breached
#        non-trivially. Not admissible as mechanism evidence — Day 15 rule.)
# 
# 8 sets × 6 conditions = 48 sims. No Δt(Q) here; that's Cell 3.
# ============================================================

import pybamm
import numpy as np
import pandas as pd
import warnings
import time as _time
from pathlib import Path

print("=== Cell 2 (rev) — Day 16 protocol feasibility, post-hoc boundary audit ===")
print(f"PyBaMM: {pybamm.__version__}")
print("DFN+thermal=lumped | set_initial_state(0.05) | Voltage events DISABLED")
print("τ_label=11.1s frozen | C-rate normalized currents")
print()

try:
    from numpy import trapezoid as _trapz
except ImportError:
    from numpy import trapz as _trapz  # noqa: NPY201

TAU_LABEL_s = 11.1
def f_from_tau_label(n_tau):
    return 1.0 / (2.0 * np.pi * n_tau * TAU_LABEL_s)

SETS_PASSING = [
    "Ai2020", "Chen2020", "Ecker2015", "Marquis2019",
    "Mohtat2020", "NCA_Kim2011", "OKane2022", "ORegan2022",
]
CHEM_TAG = {
    "Ai2020":      "Enertech LCO/graphite",
    "Chen2020":    "LG M50 lineage",
    "Ecker2015":   "Kokam-Ecker Co-rich Mn-free",
    "Marquis2019": "Kokam-Marquis LCO/graphite",
    "Mohtat2020":  "graphite/NMC532",
    "NCA_Kim2011": "graphite/NCA",
    "OKane2022":   "LG M50 lineage",
    "ORegan2022":  "LG M50 lineage",
}

CONDITIONS = [
    ("DC_0p2C",            0.2, 0.0, None),
    ("DC_0p5C",            0.5, 0.0, None),
    ("DC0p2_AC0p5_1tau",   0.2, 0.5, 1.0),
    ("DC0p2_AC0p5_10tau",  0.2, 0.5, 10.0),
    ("DC0p2_AC1p0_1tau",   0.2, 1.0, 1.0),
    ("DC0p2_AC1p0_10tau",  0.2, 1.0, 10.0),
]
VMAX_TOL = 0.005  # V

def make_current_function(I_DC_A, I_AC_A, f_Hz):
    if f_Hz is None or I_AC_A == 0.0:
        I_DC_signed = -abs(I_DC_A)
        return lambda t: I_DC_signed
    I_DC_neg = -abs(I_DC_A)
    I_AC_pos = abs(I_AC_A)
    return lambda t: I_DC_neg + I_AC_pos * np.sin(2.0 * np.pi * f_Hz * t)


records = []
total_t0 = _time.time()
voltage_events_logged = False

for set_name in SETS_PASSING:
    print(f"\n--- {set_name} ({CHEM_TAG[set_name]}) ---")
    pv0 = pybamm.ParameterValues(set_name)
    Q_nom = float(pv0["Nominal cell capacity [A.h]"])
    V_max_spec = float(pv0["Upper voltage cut-off [V]"])
    V_min_spec = float(pv0["Lower voltage cut-off [V]"])
    print(f"  Q_nom={Q_nom:.3f} Ah  V_cutoff=[{V_min_spec:.2f}, {V_max_spec:.2f}] V")

    for cond_id, DC_C, AC_C, n_tau in CONDITIONS:
        I_DC_A = DC_C * Q_nom
        I_AC_A = AC_C * Q_nom
        f_Hz = f_from_tau_label(n_tau) if n_tau is not None else None
        # 2.5× nominal CC time. Voltage-events disabled, so trajectory may
        # leave physical range — wider horizon needed.
        t_max_s = 2.5 * Q_nom / I_DC_A * 3600.0

        rec = {
            "param_set": set_name, "chem_tag": CHEM_TAG[set_name],
            "condition": cond_id,
            "DC_C": DC_C, "AC_C": AC_C,
            "tau_label": float(n_tau) if n_tau is not None else 0.0,
            "f_Hz": float(f_Hz) if f_Hz is not None else 0.0,
            "Q_nom_Ah": Q_nom,
            "I_DC_A": I_DC_A, "I_AC_A": I_AC_A,
            "Vmin_cutoff_V": V_min_spec, "Vmax_cutoff_V": V_max_spec,
            "t_max_horizon_s": t_max_s,
        }

        t0 = _time.time()
        try:
            pv = pybamm.ParameterValues(set_name)
            pv["Ambient temperature [K]"] = 293.15
            pv["Initial temperature [K]"] = 293.15
            pv.set_initial_state(0.05)
            pv["Current function [A]"] = make_current_function(I_DC_A, I_AC_A, f_Hz)

            model = pybamm.lithium_ion.DFN(options={"thermal": "lumped"})
            removed = [ev.name for ev in model.events if "voltage" in ev.name.lower()]
            model.events = [ev for ev in model.events if "voltage" not in ev.name.lower()]
            if not voltage_events_logged:
                print(f"  [info] Removed voltage events: {removed}")
                voltage_events_logged = True

            sim = pybamm.Simulation(model, parameter_values=pv)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                sol = sim.solve([0.0, t_max_s])

            t_arr = np.asarray(sol["Time [s]"].entries, dtype=float)
            V_arr = np.asarray(sol["Voltage [V]"].entries, dtype=float)
            I_arr = np.asarray(sol["Current [A]"].entries, dtype=float)

            V_init = float(V_arr[0])

            # === First Vmax crossing ===
            mask_above = V_arr >= (V_max_spec - VMAX_TOL)
            reached_Vmax = bool(mask_above.any())

            if reached_Vmax:
                idx_first = int(np.argmax(mask_above))
                # Pre-Vmax slice (inclusive)
                t_pre = t_arr[:idx_first + 1]
                V_pre = V_arr[:idx_first + 1]
                I_pre = I_arr[:idx_first + 1]
                t_to_Vmax = float(t_pre[-1])
                Q_to_Vmax = (float(_trapz(-I_pre, t_pre)) / 3600.0
                             if t_pre.size > 1 else 0.0)
                t_end_s = None
                Q_at_tmax = None
                completion = "reached_Vmax"
            else:
                t_pre = t_arr
                V_pre = V_arr
                I_pre = I_arr
                t_to_Vmax = None
                Q_to_Vmax = None
                t_end_s = float(t_arr[-1])
                Q_at_tmax = (float(_trapz(-I_arr, t_arr)) / 3600.0
                             if t_arr.size > 1 else 0.0)
                completion = "no_Vmax_by_tmax"

            # === V_min audit on pre-Vmax slice only ===
            V_min_charge = float(V_pre.min())
            V_max_charge = float(V_pre.max())
            if t_pre.size > 1:
                below = V_pre < V_min_spec
                dt = np.diff(t_pre)
                seg_below = below[:-1] | below[1:]
                t_total = float(t_pre[-1] - t_pre[0])
                frac_below = (float(np.sum(dt[seg_below]) / t_total)
                              if t_total > 0 else 0.0)
            else:
                frac_below = 0.0

            # Low-voltage class (Chen2020-anchored 0.02 reference; per-family
            # re-evaluation deferred to Cell 3)
            if V_min_charge >= V_min_spec:
                lvc = "clean"
            elif frac_below < 0.02 and (V_min_spec - V_min_charge) < 0.3:
                lvc = "transient_low_voltage"
            else:
                lvc = "severe_low_voltage"

            # Verdict
            if completion == "no_Vmax_by_tmax":
                feas = "infeasible_no_Vmax"
            elif lvc == "clean":
                feas = "feasible_clean"
            elif lvc == "transient_low_voltage":
                feas = "feasible_transient"
            else:
                feas = "boundary_confounded"

            rec.update({
                "V_init_charge": V_init,
                "V_headroom_init": V_init - V_min_spec,  # diagnostic only
                "V_min_charge": V_min_charge,
                "V_max_charge": V_max_charge,
                "V_below_Vmin_fraction": frac_below,
                "t_to_Vmax_s": t_to_Vmax,
                "Q_to_Vmax_Ah": Q_to_Vmax,
                "t_end_s": t_end_s,
                "Q_at_tmax_Ah": Q_at_tmax,
                "completion_status": completion,
                "low_voltage_class": lvc,
                "feasibility_verdict": feas,
                "solve_status": "ok",
                "error_msg": "",
            })

            t_show = (f"t→Vmax={t_to_Vmax:.0f}s" if t_to_Vmax is not None
                      else f"t_end={t_end_s:.0f}s")
            q_show = (f"Q→Vmax={Q_to_Vmax:+.3f}Ah" if Q_to_Vmax is not None
                      else f"Q@tmax={Q_at_tmax:+.3f}Ah")
            print(f"  {cond_id:<22s} V_init={V_init:.3f} V_min={V_min_charge:.3f} "
                  f"V_max={V_max_charge:.3f} {q_show} {t_show} "
                  f"frac<Vmin={frac_below:.3f} → {feas}")

        except Exception as e:
            rec.update({
                "V_init_charge": None, "V_headroom_init": None,
                "V_min_charge": None, "V_max_charge": None,
                "V_below_Vmin_fraction": None,
                "t_to_Vmax_s": None, "Q_to_Vmax_Ah": None,
                "t_end_s": None, "Q_at_tmax_Ah": None,
                "completion_status": "solver_error",
                "low_voltage_class": "solver_error",
                "feasibility_verdict": "solve_error",
                "solve_status": "fail",
                "error_msg": f"{type(e).__name__}: {str(e)[:200]}",
            })
            print(f"  {cond_id:<22s} FAIL: {rec['error_msg'][:120]}")

        rec["sim_wall_time_s"] = round(_time.time() - t0, 2)
        records.append(rec)

# === Save ===
df = pd.DataFrame(records)
out = Path("/Users/louislu/pybamm-dcac-superimposed") / "data" / "day16_step2_feasibility_matrix.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)

# === Summary by set ===
print("\n" + "=" * 100)
print("Verdict breakdown by parameter set")
print("=" * 100)
hdr = (f"  {'param_set':<14s} | {'chem':<32s} | "
       f"{'fclean':>6s} {'ftrans':>6s} {'bconf':>6s} {'noVm':>5s} {'err':>4s}")
print(hdr)
print("  " + "-" * (len(hdr) - 2))
for set_name in SETS_PASSING:
    sub = df[df["param_set"] == set_name]
    fc = int((sub["feasibility_verdict"] == "feasible_clean").sum())
    ft = int((sub["feasibility_verdict"] == "feasible_transient").sum())
    bc = int((sub["feasibility_verdict"] == "boundary_confounded").sum())
    nv = int((sub["feasibility_verdict"] == "infeasible_no_Vmax").sum())
    er = int((sub["feasibility_verdict"] == "solve_error").sum())
    print(f"  {set_name:<14s} | {CHEM_TAG[set_name]:<32s} | "
          f"{fc:>6d} {ft:>6d} {bc:>6d} {nv:>5d} {er:>4d}")

n_admissible = int(df["feasibility_verdict"].isin(
    ["feasible_clean", "feasible_transient"]).sum())
n_total = len(df)
print(f"\nCell-3 admissible (clean+transient): {n_admissible}/{n_total}")
print(f"Wall time: {round(_time.time() - total_t0, 1)}s")
print(f"\n[wrote] {out}")
print("\nNote: 'boundary_confounded' = Vmax reached but Vmin breached non-trivially.")
print("      Per Day 15 rule, NOT admissible as mechanism evidence in Cell 3.")

=== Cell 2 (rev) — Day 16 protocol feasibility, post-hoc boundary audit ===
PyBaMM: 26.3.1
DFN+thermal=lumped | set_initial_state(0.05) | Voltage events DISABLED
τ_label=11.1s frozen | C-rate normalized currents


--- Ai2020 (Enertech LCO/graphite) ---
  Q_nom=2.280 Ah  V_cutoff=[3.00, 4.20] V
  [info] Removed voltage events: ['Minimum voltage [V]', 'Maximum voltage [V]', 'Minimum voltage switch [V]', 'Maximum voltage switch [V]']
  DC_0p2C                FAIL: SolverError: IDA_LSETUP_FAIL: The linear solver's setup function failed in an unrecoverable manner


[ERROR][rank 0][/Users/runner/work/pybammsolvers/pybammsolvers/sundials/src/idas/idas.c:5738][IDAHandleFailure] At t = 19451.6942989386the linear solver setup failed unrecoverably.


  DC_0p5C                FAIL: SolverError: IDA_LSETUP_FAIL: The linear solver's setup function failed in an unrecoverable manner


[ERROR][rank 0][/Users/runner/work/pybammsolvers/pybammsolvers/sundials/src/idas/idas.c:5738][IDAHandleFailure] At t = 7703.18624281062the linear solver setup failed unrecoverably.
[ERROR][rank 0][/Users/runner/work/pybammsolvers/pybammsolvers/sundials/src/idas/idas.c:5738][IDAHandleFailure] At t = 19374.9105310873the linear solver setup failed unrecoverably.


  DC0p2_AC0p5_1tau       FAIL: SolverError: IDA_LSETUP_FAIL: The linear solver's setup function failed in an unrecoverable manner


...

  DC0p2_AC0p5_10tau      FAIL: SolverError: IDA_LSETUP_FAIL: The linear solver's setup function failed in an unrecoverable manner


[ERROR][rank 0][/Users/runner/work/pybammsolvers/pybammsolvers/sundials/src/idas/idas.c:5738][IDAHandleFailure] At t = 19374.0038512913the linear solver setup failed unrecoverably.


  DC0p2_AC1p0_1tau       FAIL: SolverError: IDA_LSETUP_FAIL: The linear solver's setup function failed in an unrecoverable manner


[ERROR][rank 0][/Users/runner/work/pybammsolvers/pybammsolvers/sundials/src/idas/idas.c:5738][IDAHandleFailure] At t = 19361.3631679801the linear solver setup failed unrecoverably.


  DC0p2_AC1p0_10tau      FAIL: SolverError: IDA_LSETUP_FAIL: The linear solver's setup function failed in an unrecoverable manner

--- Chen2020 (LG M50 lineage) ---
  Q_nom=5.000 Ah  V_cutoff=[2.50, 4.20] V
  DC_0p2C                FAIL: SolverError: IDA_ERR_FAIL: Error test failures occurred too many times during one step or minimum step size was reached
  DC_0p5C                FAIL: SolverError: IDA_ERR_FAIL: Error test failures occurred too many times during one step or minimum step size was reached
  DC0p2_AC0p5_1tau       FAIL: SolverError: IDA_ERR_FAIL: Error test failures occurred too many times during one step or minimum step size was reached
  DC0p2_AC0p5_10tau      FAIL: SolverError: IDA_ERR_FAIL: Error test failures occurred too many times during one step or minimum step size was reached
  DC0p2_AC1p0_1tau       FAIL: SolverError: IDA_ERR_FAIL: Error test failures occurred too many times during one step or minimum step size was reached
  DC0p2_AC1p0_10tau      FAIL: SolverE

In [4]:
# Cell 2-MVT — Minimum Viable Test
# Goal: validate (a) pybamm.sin fixes IDA_ERR_FAIL, 
#       (b) keeping voltage events prevents Ai2020-style stoichiometry crash.
# 
# Three configurations on Chen2020 only:
#   T1: pybamm.sin + events DISABLED + DC-only (sanity, no AC)
#   T2: pybamm.sin + events DISABLED + DC0p2_AC0p5_1tau
#   T3: pybamm.sin + events KEPT     + DC0p2_AC0p5_1tau

import pybamm
import numpy as np
import warnings

print(f"PyBaMM {pybamm.__version__}\n")

TAU_LABEL = 11.1
SET_NAME = "Chen2020"

def make_pv(set_name, I_DC, I_AC, f_Hz):
    pv = pybamm.ParameterValues(set_name)
    pv["Ambient temperature [K]"] = 293.15
    pv["Initial temperature [K]"] = 293.15
    pv.set_initial_state(0.05)
    if I_AC == 0.0 or f_Hz is None:
        I_DC_signed = -abs(I_DC)
        def i_func(t):
            return I_DC_signed
    else:
        I_DC_neg = -abs(I_DC)
        I_AC_pos = abs(I_AC)
        def i_func(t):
            # CRITICAL: pybamm.sin, NOT np.sin
            return I_DC_neg + I_AC_pos * pybamm.sin(2.0 * np.pi * f_Hz * t)
    pv["Current function [A]"] = i_func
    return pv

def make_model(disable_voltage_events):
    m = pybamm.lithium_ion.DFN(options={"thermal": "lumped"})
    if disable_voltage_events:
        m.events = [ev for ev in m.events if "voltage" not in ev.name.lower()]
    return m

def run(label, set_name, I_DC, I_AC, f_Hz, t_max, disable_events):
    print(f"--- {label} ---")
    print(f"  I_DC={I_DC:.3f}A  I_AC={I_AC:.3f}A  f={f_Hz}  t_max={t_max:.0f}s  "
          f"disable_events={disable_events}")
    try:
        pv = make_pv(set_name, I_DC, I_AC, f_Hz)
        model = make_model(disable_events)
        sim = pybamm.Simulation(model, parameter_values=pv)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            sol = sim.solve([0.0, t_max])
        t = np.asarray(sol["Time [s]"].entries, dtype=float)
        V = np.asarray(sol["Voltage [V]"].entries, dtype=float)
        I = np.asarray(sol["Current [A]"].entries, dtype=float)
        print(f"  OK  N_steps={len(t)}  t_end={t[-1]:.1f}s  "
              f"V_init={V[0]:.3f}V  V_min={V.min():.3f}V  V_max={V.max():.3f}V  "
              f"I_init={I[0]:+.3f}A  termination='{sol.termination}'")
    except Exception as e:
        print(f"  FAIL: {type(e).__name__}: {str(e)[:200]}")
    print()

Q_nom = 5.0
# T1: DC-only sanity
run("T1: DC-only, events DISABLED",
    SET_NAME, I_DC=0.2*Q_nom, I_AC=0.0, f_Hz=None,
    t_max=20000, disable_events=True)

# T2: DCAC, events disabled
f_1tau = 1.0 / (2*np.pi*TAU_LABEL)
run("T2: DCAC 1tau, events DISABLED",
    SET_NAME, I_DC=0.2*Q_nom, I_AC=0.5*Q_nom, f_Hz=f_1tau,
    t_max=20000, disable_events=True)

# T3: DCAC, events KEPT (verify natural termination at V_max)
run("T3: DCAC 1tau, events KEPT",
    SET_NAME, I_DC=0.2*Q_nom, I_AC=0.5*Q_nom, f_Hz=f_1tau,
    t_max=40000, disable_events=False)

PyBaMM 26.3.1

--- T1: DC-only, events DISABLED ---
  I_DC=1.000A  I_AC=0.000A  f=None  t_max=20000s  disable_events=True
  FAIL: SolverError: IDA_ERR_FAIL: Error test failures occurred too many times during one step or minimum step size was reached

--- T2: DCAC 1tau, events DISABLED ---
  I_DC=1.000A  I_AC=2.500A  f=0.014338283161432014  t_max=20000s  disable_events=True
  FAIL: SolverError: IDA_ERR_FAIL: Error test failures occurred too many times during one step or minimum step size was reached

--- T3: DCAC 1tau, events KEPT ---
  I_DC=1.000A  I_AC=2.500A  f=0.014338283161432014  t_max=40000s  disable_events=False
  OK  N_steps=8454  t_end=14558.7s  V_init=3.166V  V_min=3.013V  V_max=4.200V  I_init=-1.000A  termination='event: Maximum voltage [V]'



In [7]:
# ============================================================
# Cell 2 rev3 — Day 16 protocol feasibility (events kept, pybamm.sin)
#               + trajectory persistence for Cell 3A
# 
# Verdict logic identical to rev2.
# Adds: per-case (t, V, I) saved to npz so Cell 3A computes Δt(Q) without
# re-simulating.
# 
# Outputs:
#   data/day16_step2_feasibility_matrix_rev3.csv
#   data/day16_step2_trajectories_rev3.npz
# ============================================================

import pybamm
import numpy as np
import pandas as pd
import warnings
import time as _time
from pathlib import Path

print("=== Cell 2 rev3 — Day 16 feasibility + trajectory persistence ===")
print(f"PyBaMM: {pybamm.__version__}")
print("DFN+thermal=lumped | set_initial_state(0.05) | model.events KEPT")
print("τ_label=11.1s frozen | C-rate normalized | pv callable + pybamm.sin")
print()

try:
    from numpy import trapezoid as _trapz
except ImportError:
    from numpy import trapz as _trapz  # noqa: NPY201

TAU_LABEL_s = 11.1
def f_from_tau_label(n_tau):
    return 1.0 / (2.0 * np.pi * n_tau * TAU_LABEL_s)

SETS_PASSING = [
    "Ai2020", "Chen2020", "Ecker2015", "Marquis2019",
    "Mohtat2020", "NCA_Kim2011", "OKane2022", "ORegan2022",
]
CHEM_TAG = {
    "Ai2020":      "Enertech LCO/graphite",
    "Chen2020":    "LG M50 lineage",
    "Ecker2015":   "Kokam-Ecker Co-rich Mn-free",
    "Marquis2019": "Kokam-Marquis LCO/graphite",
    "Mohtat2020":  "graphite/NMC532",
    "NCA_Kim2011": "graphite/NCA",
    "OKane2022":   "LG M50 lineage",
    "ORegan2022":  "LG M50 lineage",
}

CONDITIONS = [
    ("DC_0p2C",            0.2, 0.0, None),
    ("DC_0p5C",            0.5, 0.0, None),
    ("DC0p2_AC0p5_1tau",   0.2, 0.5, 1.0),
    ("DC0p2_AC0p5_10tau",  0.2, 0.5, 10.0),
    ("DC0p2_AC1p0_1tau",   0.2, 1.0, 1.0),
    ("DC0p2_AC1p0_10tau",  0.2, 1.0, 10.0),
]

def make_current_function(I_DC_A, I_AC_A, f_Hz):
    if f_Hz is None or I_AC_A == 0.0:
        I_DC_signed = -abs(I_DC_A)
        def i_func(t):
            return I_DC_signed
        return i_func
    I_DC_neg = -abs(I_DC_A)
    I_AC_pos = abs(I_AC_A)
    def i_func(t):
        return I_DC_neg + I_AC_pos * pybamm.sin(2.0 * np.pi * f_Hz * t)
    return i_func


def classify_termination(termination_str):
    s = (termination_str or "").lower()
    if "maximum voltage" in s:
        return "Vmax_event"
    if "minimum voltage" in s:
        return "Vmin_event"
    if "final time" in s:
        return "tmax_horizon"
    return f"other:{s[:40]}"


records = []
trajectories = {}  # key=f"{set}__{cond}" → dict of arrays
total_t0 = _time.time()

for set_name in SETS_PASSING:
    print(f"\n--- {set_name} ({CHEM_TAG[set_name]}) ---")
    pv0 = pybamm.ParameterValues(set_name)
    Q_nom = float(pv0["Nominal cell capacity [A.h]"])
    V_max_spec = float(pv0["Upper voltage cut-off [V]"])
    V_min_spec = float(pv0["Lower voltage cut-off [V]"])
    print(f"  Q_nom={Q_nom:.3f} Ah  V_cutoff=[{V_min_spec:.2f}, {V_max_spec:.2f}] V")

    for cond_id, DC_C, AC_C, n_tau in CONDITIONS:
        I_DC_A = DC_C * Q_nom
        I_AC_A = AC_C * Q_nom
        f_Hz = f_from_tau_label(n_tau) if n_tau is not None else None
        t_max_s = 2.5 * Q_nom / I_DC_A * 3600.0

        rec = {
            "param_set": set_name, "chem_tag": CHEM_TAG[set_name],
            "condition": cond_id,
            "DC_C": DC_C, "AC_C": AC_C,
            "tau_label": float(n_tau) if n_tau is not None else 0.0,
            "f_Hz": float(f_Hz) if f_Hz is not None else 0.0,
            "Q_nom_Ah": Q_nom,
            "I_DC_A": I_DC_A, "I_AC_A": I_AC_A,
            "Vmin_cutoff_V": V_min_spec,
            "Vmax_cutoff_V": V_max_spec,
            "t_max_horizon_s": t_max_s,
        }

        t0 = _time.time()
        try:
            pv = pybamm.ParameterValues(set_name)
            pv["Ambient temperature [K]"] = 293.15
            pv["Initial temperature [K]"] = 293.15
            pv.set_initial_state(0.05)
            pv["Current function [A]"] = make_current_function(I_DC_A, I_AC_A, f_Hz)

            model = pybamm.lithium_ion.DFN(options={"thermal": "lumped"})
            sim = pybamm.Simulation(model, parameter_values=pv)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                sol = sim.solve([0.0, t_max_s])

            t_arr = np.asarray(sol["Time [s]"].entries, dtype=float)
            V_arr = np.asarray(sol["Voltage [V]"].entries, dtype=float)
            I_arr = np.asarray(sol["Current [A]"].entries, dtype=float)

            # Persist trajectory for Cell 3A
            traj_key = f"{set_name}__{cond_id}"
            trajectories[traj_key] = {
                "t_s": t_arr.copy(), "V_V": V_arr.copy(), "I_A": I_arr.copy(),
            }

            term_raw = str(getattr(sol, "termination", ""))
            term_class = classify_termination(term_raw)

            V_init = float(V_arr[0])
            V_min_charge = float(V_arr.min())
            V_max_charge = float(V_arr.max())
            t_end = float(t_arr[-1])

            if t_arr.size > 1:
                below = V_arr < V_min_spec
                dt = np.diff(t_arr)
                seg_below = below[:-1] | below[1:]
                t_total = float(t_arr[-1] - t_arr[0])
                frac_below = (float(np.sum(dt[seg_below]) / t_total)
                              if t_total > 0 else 0.0)
                Q_net_Ah = float(_trapz(-I_arr, t_arr)) / 3600.0
            else:
                frac_below = 0.0
                Q_net_Ah = 0.0

            if V_min_charge >= V_min_spec:
                lvc = "clean"
            elif frac_below < 0.02 and (V_min_spec - V_min_charge) < 0.3:
                lvc = "transient_low_voltage"
            else:
                lvc = "severe_low_voltage"

            if term_class == "Vmax_event":
                if lvc == "clean":
                    feas = "feasible_clean"
                elif lvc == "transient_low_voltage":
                    feas = "feasible_transient"
                else:
                    feas = "boundary_confounded"
            elif term_class == "Vmin_event":
                feas = "infeasible_Vmin_event"
            elif term_class == "tmax_horizon":
                feas = "infeasible_no_Vmax"
            else:
                feas = f"unexpected:{term_class}"

            rec.update({
                "V_init_charge": V_init,
                "V_headroom_init": V_init - V_min_spec,
                "V_min_charge": V_min_charge,
                "V_max_charge": V_max_charge,
                "V_below_Vmin_fraction": frac_below,
                "Q_CC_end_Ah": Q_net_Ah,
                "t_charge_s": t_end,
                "termination_raw": term_raw[:120],
                "termination_class": term_class,
                "low_voltage_class": lvc,
                "feasibility_verdict": feas,
                "solve_status": "ok",
                "error_msg": "",
            })

            print(f"  {cond_id:<22s} V_init={V_init:.3f} V_min={V_min_charge:.3f} "
                  f"V_max={V_max_charge:.3f} Q={Q_net_Ah:+.3f}Ah t={t_end:.0f}s "
                  f"frac<Vmin={frac_below:.3f} → {feas} [{term_class}]")

        except Exception as e:
            rec.update({
                "V_init_charge": None, "V_headroom_init": None,
                "V_min_charge": None, "V_max_charge": None,
                "V_below_Vmin_fraction": None,
                "Q_CC_end_Ah": None, "t_charge_s": None,
                "termination_raw": "", "termination_class": "solve_error",
                "low_voltage_class": "solve_error",
                "feasibility_verdict": "solve_error",
                "solve_status": "fail",
                "error_msg": f"{type(e).__name__}: {str(e)[:200]}",
            })
            print(f"  {cond_id:<22s} FAIL: {rec['error_msg'][:120]}")

        rec["sim_wall_time_s"] = round(_time.time() - t0, 2)
        records.append(rec)

# === Save CSV ===
df = pd.DataFrame(records)
out_csv = Path("/Users/louislu/pybamm-dcac-superimposed") / "data" / "day16_step2_feasibility_matrix_rev3.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_csv, index=False)

# === Save trajectories npz (flat-dict layout) ===
out_npz = Path("/Users/louislu/pybamm-dcac-superimposed") / "data" / "day16_step2_trajectories_rev3.npz"
flat = {}
for k, v in trajectories.items():
    flat[f"{k}__t_s"] = v["t_s"]
    flat[f"{k}__V_V"] = v["V_V"]
    flat[f"{k}__I_A"] = v["I_A"]
np.savez_compressed(out_npz, **flat)

# === Summary ===
print("\n" + "=" * 110)
print("Verdict breakdown by parameter set (rev3: events kept, pybamm.sin, trajectories saved)")
print("=" * 110)
hdr = (f"  {'param_set':<14s} | {'chem':<32s} | "
       f"{'fclean':>6s} {'ftrans':>6s} {'bconf':>6s} {'iVmin':>5s} {'inoVm':>5s} {'err':>4s}")
print(hdr)
print("  " + "-" * (len(hdr) - 2))
for set_name in SETS_PASSING:
    sub = df[df["param_set"] == set_name]
    fc = int((sub["feasibility_verdict"] == "feasible_clean").sum())
    ft = int((sub["feasibility_verdict"] == "feasible_transient").sum())
    bc = int((sub["feasibility_verdict"] == "boundary_confounded").sum())
    iv = int((sub["feasibility_verdict"] == "infeasible_Vmin_event").sum())
    nv = int((sub["feasibility_verdict"] == "infeasible_no_Vmax").sum())
    er = int((sub["feasibility_verdict"] == "solve_error").sum())
    print(f"  {set_name:<14s} | {CHEM_TAG[set_name]:<32s} | "
          f"{fc:>6d} {ft:>6d} {bc:>6d} {iv:>5d} {nv:>5d} {er:>4d}")

n_admissible = int(df["feasibility_verdict"].isin(
    ["feasible_clean", "feasible_transient"]).sum())
print(f"\nCell-3 admissible (clean+transient): {n_admissible}/{len(df)}")
print(f"Trajectories saved: {len(trajectories)}/{len(df)}")
print(f"Wall time: {round(_time.time() - total_t0, 1)}s")
print(f"\n[wrote] {out_csv}")
print(f"[wrote] {out_npz}  ({out_npz.stat().st_size / 1024 / 1024:.2f} MB)")

=== Cell 2 rev3 — Day 16 feasibility + trajectory persistence ===
PyBaMM: 26.3.1
DFN+thermal=lumped | set_initial_state(0.05) | model.events KEPT
τ_label=11.1s frozen | C-rate normalized | pv callable + pybamm.sin


--- Ai2020 (Enertech LCO/graphite) ---
  Q_nom=2.280 Ah  V_cutoff=[3.00, 4.20] V
  DC_0p2C                V_init=3.586 V_min=3.586 V_max=4.200 Q=+2.290Ah t=18079s frac<Vmin=0.000 → feasible_clean [Vmax_event]
  DC_0p5C                V_init=3.636 V_min=3.636 V_max=4.200 Q=+2.194Ah t=6927s frac<Vmin=0.000 → feasible_clean [Vmax_event]
  DC0p2_AC0p5_1tau       V_init=3.586 V_min=3.474 V_max=4.200 Q=+2.166Ah t=17138s frac<Vmin=0.000 → feasible_clean [Vmax_event]
  DC0p2_AC0p5_10tau      V_init=3.586 V_min=3.422 V_max=4.200 Q=+2.148Ah t=17252s frac<Vmin=0.000 → feasible_clean [Vmax_event]
  DC0p2_AC1p0_1tau       V_init=3.586 V_min=3.371 V_max=4.200 Q=+2.063Ah t=16371s frac<Vmin=0.000 → feasible_clean [Vmax_event]
  DC0p2_AC1p0_10tau      V_init=3.586 V_min=3.040 V_max=4.200 Q=

In [8]:
# ============================================================
# Cell 3A (rev) — Day 16 Δt(Q) first-pass calculation, sign summary
# 
# Inputs:
#   data/day16_step2_feasibility_matrix_rev3.csv
#   data/day16_step2_trajectories_rev3.npz
# 
# Direction (Day 13/14 convention):
#   dtQ_s = t_ref_DC_s - t_protocol_s
#   > 0  →  DCAC reaches the same Q earlier than DC reference
#   = 0  →  equivalent (DC sanity rows must give exactly this)
#   < 0  →  DCAC slower than DC
# 
# Pairing rules (frozen):
#   - reference  = same parameter_set + DC_0p2C
#   - protocol   = same parameter_set + DCAC condition
#   - DC sanity  = DC_0p2C vs itself (validation row, dtQ ≡ 0)
#   - DC_0p5C is not used (different DC_C from DCAC conditions)
# 
# Window (frozen):
#   Q_low = 0.05 × Q_nom
#   Q_hi  = min(Q_to_Vmax_protocol, Q_to_Vmax_DC_ref) − 0.02 × Q_nom
# 
# A_Δt scalars (units explicit):
#   A_delta_t_area_sAh = ∫ dtQ dQ           (units: s·Ah)
#   A_delta_t_mean_s   = area / Q_span      (units: s, window-averaged Δt)
# 
# Outputs:
#   data/day16_step3a_dtQ_table.csv             (36 rows summary)
#   data/day16_step3a_dtQ_curves_long.csv.gz    (long-format curves for Cell 3B/4)
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/Users/louislu/pybamm-dcac-superimposed") / "data"
CSV_IN = DATA_DIR / "day16_step2_feasibility_matrix_rev3.csv"
NPZ_IN = DATA_DIR / "day16_step2_trajectories_rev3.npz"
OUT = DATA_DIR / "day16_step3a_dtQ_table.csv"
OUT_CURVES = DATA_DIR / "day16_step3a_dtQ_curves_long.csv.gz"

print("=== Cell 3A (rev) — Day 16 Δt(Q) first-pass calculation ===\n")
df_feas = pd.read_csv(CSV_IN)
npz = np.load(NPZ_IN)
print(f"Feasibility matrix: {len(df_feas)} rows")
print(f"Trajectory arrays:  {len(npz.files) // 3} cases\n")


def load_traj(set_name, cond):
    k = f"{set_name}__{cond}"
    return {"t_s": npz[f"{k}__t_s"],
            "V_V": npz[f"{k}__V_V"],
            "I_A": npz[f"{k}__I_A"]}


def Q_net_cum(t, I):
    """Cumulative net charge (Ah). Charge throughput = ∫(-I)dt / 3600.
    Trapezoidal rule, matches Cell 2 Q_CC_end formula."""
    if len(t) < 2:
        return np.zeros_like(t)
    dt = np.diff(t)
    avg = 0.5 * (-I[:-1] - I[1:])
    return np.concatenate([[0.0], np.cumsum(avg * dt) / 3600.0])


def Q_to_Vmax_exact(t, V, I, V_max_spec, term_class):
    """If termination is Vmax_event, sundials root-finds to event boundary,
    so trajectory endpoint Q is the precise first-passage Q. Otherwise
    fallback to V threshold (V_max - 5mV) interpolation."""
    Q = Q_net_cum(t, I)
    if term_class == "Vmax_event":
        return float(Q[-1])
    target_V = V_max_spec - 0.005
    mask = V >= target_V
    if not mask.any():
        return np.nan
    idx = int(np.argmax(mask))
    if idx == 0:
        return float(Q[0])
    V0, V1 = V[idx-1], V[idx]
    Q0, Q1 = Q[idx-1], Q[idx]
    if V1 == V0:
        return float(Q1)
    return float(Q0 + (target_V - V0) * (Q1 - Q0) / (V1 - V0))


def t_at_Q_first_passage(t, Q_cum, Q_target):
    """Linear interpolation for first-passage time at Q_target."""
    mask = Q_cum >= Q_target
    if not mask.any():
        return np.nan
    idx = int(np.argmax(mask))
    if idx == 0:
        return float(t[0])
    Q0, Q1 = Q_cum[idx-1], Q_cum[idx]
    t0, t1 = t[idx-1], t[idx]
    if Q1 == Q0:
        return float(t1)
    return float(t0 + (Q_target - Q0) * (t1 - t0) / (Q1 - Q0))


def class_abs(max_abs, frac_pos):
    """Chen2020-anchored absolute threshold (Day 14 historical).
    Near-zero threshold: |dtQ|_max < 60 s (absolute, not normalized)."""
    if max_abs < 60.0:
        return "near_zero"
    if frac_pos >= 0.95:
        return "positive_only"
    if frac_pos <= 0.05:
        return "negative_only"
    return "mixed"


def class_rel(max_abs, frac_pos, t_charge_DC_ref):
    """Cross-set fair sign-class.
    Near-zero threshold: 0.5% of same-set DC_0p2C time-to-Vmax
    (NOT window time)."""
    if t_charge_DC_ref is None or t_charge_DC_ref <= 0:
        return "ref_invalid"
    if np.isnan(t_charge_DC_ref):
        return "ref_invalid"
    if max_abs < 0.005 * t_charge_DC_ref:
        return "near_zero"
    if frac_pos >= 0.95:
        return "positive_only"
    if frac_pos <= 0.05:
        return "negative_only"
    return "mixed"


SETS = ["Ai2020", "Chen2020", "Ecker2015", "Marquis2019",
        "Mohtat2020", "NCA_Kim2011", "OKane2022", "ORegan2022"]
ADMISSIBLE = {"feasible_clean", "feasible_transient"}
N_GRID = 100

records = []
curve_records = []  # long-format Δt(Q) curves for Cell 3B/4

for set_name in SETS:
    # Reference: DC_0p2C
    ref_rows = df_feas[(df_feas["param_set"] == set_name) &
                       (df_feas["condition"] == "DC_0p2C")]
    if len(ref_rows) != 1:
        print(f"[{set_name}] missing DC_0p2C reference, skip")
        continue
    ref_row = ref_rows.iloc[0]
    if ref_row["feasibility_verdict"] not in ADMISSIBLE:
        print(f"[{set_name}] DC_0p2C reference not admissible, skip set")
        continue

    Q_nom = float(ref_row["Q_nom_Ah"])
    chem_tag = ref_row["chem_tag"]
    V_max_spec = float(ref_row["Vmax_cutoff_V"])
    ref_term_class = ref_row["termination_class"]

    ref_traj = load_traj(set_name, "DC_0p2C")
    t_ref, V_ref, I_ref = ref_traj["t_s"], ref_traj["V_V"], ref_traj["I_A"]
    Q_ref_cum = Q_net_cum(t_ref, I_ref)
    Q_to_Vmax_DC = Q_to_Vmax_exact(t_ref, V_ref, I_ref, V_max_spec, ref_term_class)
    t_charge_DC_ref = float(t_ref[-1])

    print(f"\n[{set_name}]  Q_nom={Q_nom:.3f}Ah  "
          f"Q→Vmax(DC_ref)={Q_to_Vmax_DC:.3f}Ah  "
          f"t_charge(DC_ref)={t_charge_DC_ref:.0f}s")

    sub = df_feas[df_feas["param_set"] == set_name]

    for _, prow in sub.iterrows():
        cond = prow["condition"]
        verdict = prow["feasibility_verdict"]

        # Skip DC_0p5C (not part of pairing)
        if cond == "DC_0p5C":
            continue

        # Pair role
        if cond == "DC_0p2C":
            pair_role = "DC_sanity"
        else:
            pair_role = "DCAC"
            if verdict not in ADMISSIBLE:
                continue  # excluded; recorded in Cell 2 CSV

        # Load protocol trajectory
        try:
            p_traj = load_traj(set_name, cond)
        except KeyError:
            print(f"  {cond:<22s} trajectory missing, skip")
            continue

        t_p, V_p, I_p = p_traj["t_s"], p_traj["V_V"], p_traj["I_A"]
        Q_p_cum = Q_net_cum(t_p, I_p)
        p_term_class = prow["termination_class"]
        Q_to_Vmax_proto = Q_to_Vmax_exact(t_p, V_p, I_p, V_max_spec, p_term_class)

        Q_low = 0.05 * Q_nom
        Q_hi = min(Q_to_Vmax_proto, Q_to_Vmax_DC) - 0.02 * Q_nom

        rec = {
            "param_set": set_name,
            "chem_tag": chem_tag,
            "condition": cond,
            "pair_role": pair_role,
            "DC_C": float(prow["DC_C"]),
            "AC_C": float(prow["AC_C"]),
            "tau_label": float(prow["tau_label"]),
            "Q_nom_Ah": Q_nom,
            "Q_to_Vmax_protocol_Ah": Q_to_Vmax_proto,
            "Q_to_Vmax_DC_ref_Ah": Q_to_Vmax_DC,
            "Q_low_Ah": Q_low,
            "Q_hi_Ah": Q_hi,
            "window_width_Ah": Q_hi - Q_low,
            "t_charge_DC_ref_s": t_charge_DC_ref,
            "feasibility_verdict_protocol": verdict,
            "feasibility_verdict_DC_ref": ref_row["feasibility_verdict"],
            "low_voltage_class_protocol": prow["low_voltage_class"],
            "low_voltage_class_DC_ref": ref_row["low_voltage_class"],
        }

        nan_metric_keys = [
            "dtQ_min_s", "dtQ_max_s", "dtQ_mean_s", "dtQ_median_s",
            "dtQ_max_abs_s", "n_pos", "n_neg", "n_zero",
            "dtQ_fraction_positive",
            "A_delta_t_area_sAh", "A_delta_t_mean_s",
        ]

        if Q_hi <= Q_low or np.isnan(Q_hi) or np.isnan(Q_low):
            rec["window_status"] = "collapsed"
            rec["n_Q_grid"] = 0
            for k in nan_metric_keys:
                rec[k] = np.nan
            rec["sign_class_abs"] = "window_collapsed"
            rec["sign_class_rel"] = "window_collapsed"
            records.append(rec)
            print(f"  {cond:<22s} window collapsed "
                  f"(Q_low={Q_low:.3f} ≥ Q_hi={Q_hi:.3f})")
            continue

        rec["window_status"] = "ok"
        rec["n_Q_grid"] = N_GRID
        Q_grid = np.linspace(Q_low, Q_hi, N_GRID)

        t_DC_at = np.array([t_at_Q_first_passage(t_ref, Q_ref_cum, q) for q in Q_grid])
        t_p_at = np.array([t_at_Q_first_passage(t_p, Q_p_cum, q) for q in Q_grid])
        dtQ = t_DC_at - t_p_at  # > 0 = DCAC faster

        # Persist long-format curve points for Cell 3B / Cell 4
        for q, tdc, tp, dt in zip(Q_grid, t_DC_at, t_p_at, dtQ):
            curve_records.append({
                "param_set": set_name,
                "chem_tag": chem_tag,
                "condition": cond,
                "pair_role": pair_role,
                "DC_C": float(prow["DC_C"]),
                "AC_C": float(prow["AC_C"]),
                "tau_label": float(prow["tau_label"]),
                "Q_Ah": float(q),
                "t_DC_s": float(tdc) if np.isfinite(tdc) else np.nan,
                "t_protocol_s": float(tp) if np.isfinite(tp) else np.nan,
                "dtQ_s": float(dt) if np.isfinite(dt) else np.nan,
                "Q_nom_Ah": Q_nom,
                "Q_low_Ah": Q_low,
                "Q_hi_Ah": Q_hi,
            })

        finite = np.isfinite(dtQ)
        if not finite.any():
            rec["window_status"] = "all_nan"
            for k in nan_metric_keys:
                rec[k] = np.nan
            rec["sign_class_abs"] = "all_nan"
            rec["sign_class_rel"] = "all_nan"
            records.append(rec)
            print(f"  {cond:<22s} all NaN")
            continue

        dtQ_f = dtQ[finite]
        Q_f = Q_grid[finite]
        rec["dtQ_min_s"] = float(np.min(dtQ_f))
        rec["dtQ_max_s"] = float(np.max(dtQ_f))
        rec["dtQ_mean_s"] = float(np.mean(dtQ_f))
        rec["dtQ_median_s"] = float(np.median(dtQ_f))
        max_abs = float(np.max(np.abs(dtQ_f)))
        rec["dtQ_max_abs_s"] = max_abs
        n_pos = int((dtQ_f > 0).sum())
        n_neg = int((dtQ_f < 0).sum())
        n_zero = int((dtQ_f == 0).sum())
        rec["n_pos"] = n_pos
        rec["n_neg"] = n_neg
        rec["n_zero"] = n_zero
        n_tot = n_pos + n_neg + n_zero
        frac_pos = n_pos / n_tot if n_tot > 0 else np.nan
        rec["dtQ_fraction_positive"] = frac_pos

        # A_Δt: two flavors, units explicit
        # area = ∫ dtQ dQ        (units: s·Ah)
        # mean = area / Q_span   (units: s, window-averaged Δt)
        Q_span = Q_f[-1] - Q_f[0]
        try:
            A_dt_area = float(np.trapezoid(dtQ_f, Q_f))
        except AttributeError:
            A_dt_area = float(np.trapz(dtQ_f, Q_f))  # noqa: NPY201
        rec["A_delta_t_area_sAh"] = A_dt_area
        rec["A_delta_t_mean_s"] = A_dt_area / Q_span if Q_span > 0 else np.nan

        rec["sign_class_abs"] = class_abs(max_abs, frac_pos)
        rec["sign_class_rel"] = class_rel(max_abs, frac_pos, t_charge_DC_ref)

        records.append(rec)
        marker = " (sanity)" if pair_role == "DC_sanity" else ""
        print(f"  {cond:<22s}{marker:<9s}  "
              f"Q_win=[{Q_low:.3f}, {Q_hi:.3f}]Ah  "
              f"dtQ ∈ [{rec['dtQ_min_s']:+.2f}, {rec['dtQ_max_s']:+.2f}]s  "
              f"|max|={max_abs:.2f}s  "
              f"A_area={rec['A_delta_t_area_sAh']:+.3f}s·Ah "
              f"A_mean={rec['A_delta_t_mean_s']:+.3f}s  "
              f"abs={rec['sign_class_abs']} rel={rec['sign_class_rel']}")

# === Save summary table ===
df_out = pd.DataFrame(records)
df_out.to_csv(OUT, index=False)

# === Save long-format curves ===
df_curves = pd.DataFrame(curve_records)
df_curves.to_csv(OUT_CURVES, index=False, compression="gzip")

# === Summary ===
print("\n" + "=" * 100)
print("Cell 3A summary")
print("=" * 100)
print(f"Total summary rows: {len(df_out)}")
n_dcac = int((df_out["pair_role"] == "DCAC").sum())
n_sanity = int((df_out["pair_role"] == "DC_sanity").sum())
print(f"  DCAC rows:   {n_dcac}")
print(f"  Sanity rows: {n_sanity}")
print(f"Total curve points: {len(df_curves)}")

# DC sanity check — must be ≈ 0 within numerical noise
sanity = df_out[df_out["pair_role"] == "DC_sanity"]
if len(sanity):
    finite_max = sanity["dtQ_max_abs_s"].dropna().max()
    status = "OK" if (finite_max < 1e-6) else "BUG"
    print(f"\nDC sanity max |dtQ|: {finite_max:.2e}s  [{status}: should be 0; "
          f">1e-6 indicates interpolation bug]")

# Sign-class breakdowns
print("\nSign class (abs, Chen2020-anchored 60s threshold) — DCAC rows:")
print(df_out[df_out["pair_role"] == "DCAC"]["sign_class_abs"].value_counts().to_string())
print("\nSign class (rel, 0.5% of DC reference t_charge) — DCAC rows:")
print(df_out[df_out["pair_role"] == "DCAC"]["sign_class_rel"].value_counts().to_string())

# Cross-classification (where do the two thresholds disagree?)
print("\nAbs vs Rel disagreement (DCAC rows where the two classes differ):")
dcac = df_out[df_out["pair_role"] == "DCAC"]
disagree = dcac[dcac["sign_class_abs"] != dcac["sign_class_rel"]]
if len(disagree):
    cols = ["param_set", "condition", "Q_nom_Ah",
            "dtQ_max_abs_s", "t_charge_DC_ref_s",
            "sign_class_abs", "sign_class_rel"]
    print(disagree[cols].to_string(index=False))
else:
    print("  (none — abs and rel agree on all DCAC rows)")

print(f"\n[wrote] {OUT}")
print(f"[wrote] {OUT_CURVES}  ({df_curves.shape[0]} points, "
      f"{OUT_CURVES.stat().st_size / 1024:.1f} KB compressed)")

=== Cell 3A (rev) — Day 16 Δt(Q) first-pass calculation ===

Feasibility matrix: 48 rows
Trajectory arrays:  48 cases


[Ai2020]  Q_nom=2.280Ah  Q→Vmax(DC_ref)=2.290Ah  t_charge(DC_ref)=18079s
  DC_0p2C                (sanity)  Q_win=[0.114, 2.244]Ah  dtQ ∈ [+0.00, +0.00]s  |max|=0.00s  A_area=+0.000s·Ah A_mean=+0.000s  abs=near_zero rel=near_zero
  DC0p2_AC0p5_1tau                 Q_win=[0.114, 2.120]Ah  dtQ ∈ [-51.25, -1.94]s  |max|=51.25s  A_area=-49.552s·Ah A_mean=-24.702s  abs=near_zero rel=near_zero
  DC0p2_AC0p5_10tau                Q_win=[0.114, 2.102]Ah  dtQ ∈ [-418.07, +0.98]s  |max|=418.07s  A_area=-373.836s·Ah A_mean=-188.009s  abs=negative_only rel=negative_only
  DC0p2_AC1p0_1tau                 Q_win=[0.114, 2.017]Ah  dtQ ∈ [-74.33, -2.91]s  |max|=74.33s  A_area=-72.914s·Ah A_mean=-38.312s  abs=negative_only rel=near_zero
  DC0p2_AC1p0_10tau                Q_win=[0.114, 1.975]Ah  dtQ ∈ [-521.15, -1.62]s  |max|=521.15s  A_area=-455.809s·Ah A_mean=-244.950s  abs=negative_o

In [10]:
# ============================================================
# Cell 3B — Day 16 visualization (5 figures)
# 
# Inputs:
#   data/day16_step2_feasibility_matrix_rev3.csv   (for B5 — includes infeasible)
#   data/day16_step3a_dtQ_table.csv                (for B1, B2, B3 — summary)
#   data/day16_step3a_dtQ_curves_long.csv.gz       (for B4 — Δt(Q) curves)
# 
# Outputs (saved to figures/day16/):
#   B1_scatter_overview.png       28 cases at a glance
#   B2_heatmap_Amean.png          A_delta_t_mean_s per (set, condition)
#   B3_heatmap_signclass.png      sign_class_rel per (set, condition)
#   B4_curves_AC0p5_10tau.png     8 sets overlaid at DC0p2_AC0p5_10tau
#   B5_feasibility_envelope.png   feasibility verdict per (set, all 6 conditions)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path

DATA_DIR = Path("/Users/louislu/pybamm-dcac-superimposed") / "data"
FIG_DIR = Path("/Users/louislu/pybamm-dcac-superimposed") / "figures" / "day16"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("=== Cell 3B — Day 16 visualization (5 figures) ===\n")

df_feas = pd.read_csv(DATA_DIR / "day16_step2_feasibility_matrix_rev3.csv")
df_table = pd.read_csv(DATA_DIR / "day16_step3a_dtQ_table.csv")
df_curves = pd.read_csv(DATA_DIR / "day16_step3a_dtQ_curves_long.csv.gz")

print(f"Feasibility rows: {len(df_feas)}")
print(f"Δt(Q) summary rows: {len(df_table)}  ({(df_table['pair_role']=='DCAC').sum()} DCAC + "
      f"{(df_table['pair_role']=='DC_sanity').sum()} sanity)")
print(f"Δt(Q) curve points: {len(df_curves)}\n")

# === Common: parameter set order (group by chemistry family for visual coherence) ===
SET_ORDER = [
    "Ai2020", "Marquis2019",                    # LCO
    "Chen2020", "OKane2022", "ORegan2022",      # LG M50 lineage (NMC811-Si)
    "Mohtat2020",                                # NMC532
    "NCA_Kim2011",                               # NCA
    "Ecker2015",                                 # Kokam-Ecker
]
CHEM_FAMILY = {
    "Ai2020": "LCO",
    "Marquis2019": "LCO",
    "Chen2020": "LG M50",
    "OKane2022": "LG M50",
    "ORegan2022": "LG M50",
    "Mohtat2020": "NMC532",
    "NCA_Kim2011": "NCA",
    "Ecker2015": "Kokam-Ecker",
}
# Color per family (5 families)
FAMILY_COLOR = {
    "LCO":         "#d62728",  # red
    "LG M50":      "#1f77b4",  # blue
    "NMC532":      "#2ca02c",  # green
    "NCA":         "#9467bd",  # purple
    "Kokam-Ecker": "#ff7f0e",  # orange
}
# Marker per parameter set (within family, distinguish Ai/Marquis and the 3 LG M50)
SET_MARKER = {
    "Ai2020": "o",         "Marquis2019": "s",
    "Chen2020": "o",       "OKane2022": "^",     "ORegan2022": "v",
    "Mohtat2020": "o",
    "NCA_Kim2011": "o",
    "Ecker2015": "o",
}

CONDITION_ORDER = [
    "DC_0p2C", "DC_0p5C",
    "DC0p2_AC0p5_1tau", "DC0p2_AC0p5_10tau",
    "DC0p2_AC1p0_1tau", "DC0p2_AC1p0_10tau",
]
CONDITION_LABEL = {
    "DC_0p2C":            "DC 0.2C",
    "DC_0p5C":            "DC 0.5C",
    "DC0p2_AC0p5_1tau":   "0.2+0.5C\n1τ",
    "DC0p2_AC0p5_10tau":  "0.2+0.5C\n10τ",
    "DC0p2_AC1p0_1tau":   "0.2+1.0C\n1τ",
    "DC0p2_AC1p0_10tau":  "0.2+1.0C\n10τ",
}

# ==========================================================================
# Figure B1 — scatter overview: AC_C (x) × A_mean (y), color = tau, marker = set
# ==========================================================================
print("Generating B1 scatter overview ...")
dcac = df_table[df_table["pair_role"] == "DCAC"].copy()

fig, ax = plt.subplots(figsize=(10, 6.5))

# Slight x-jitter so 1τ and 10τ don't overlap exactly at AC=0.5 / AC=1.0
np.random.seed(0)
for _, row in dcac.iterrows():
    set_name = row["param_set"]
    fam = CHEM_FAMILY[set_name]
    tau = row["tau_label"]
    # x position: AC_C with small jitter by tau (1τ left, 10τ right)
    x_jitter = -0.025 if tau == 1.0 else +0.025
    x = row["AC_C"] + x_jitter
    y = row["A_delta_t_mean_s"]
    color = FAMILY_COLOR[fam]
    marker = SET_MARKER[set_name]
    edge = "black" if tau == 10.0 else color
    lw = 1.5 if tau == 10.0 else 0.5
    ax.scatter(x, y, c=color, marker=marker, s=140,
               edgecolors=edge, linewidths=lw, alpha=0.85, zorder=3)

ax.axhline(0, color="black", linewidth=0.8, linestyle="-", zorder=1)
ax.axhline(-60, color="gray", linewidth=0.6, linestyle=":", zorder=1)
ax.axhline(+60, color="gray", linewidth=0.6, linestyle=":", zorder=1)
ax.text(1.07, -60, "abs near_zero ±60s", fontsize=8, va="center", color="gray")

ax.set_xlabel("AC amplitude [C-rate]", fontsize=11)
ax.set_ylabel(r"$A_{\Delta t,\mathrm{mean}}$  [s]   (>0 = DCAC faster)", fontsize=11)
ax.set_title("B1: Window-averaged Δt(Q) across 28 admissible DC–AC pairs\n"
             "(jitter: 1τ left, 10τ right;  marker edge bold = 10τ)",
             fontsize=11)
ax.set_xticks([0.5, 1.0])
ax.set_xticklabels(["0.5C", "1.0C"])
ax.grid(True, alpha=0.3, zorder=0)

# Family-color legend
fam_handles = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLOR.items()]
leg1 = ax.legend(handles=fam_handles, loc="lower right", fontsize=9,
                 title="chemistry family", framealpha=0.9)
ax.add_artist(leg1)

# Marker legend (LG M50 lineage internal)
m_handles = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor="gray",
               markersize=10, label="primary"),
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor="gray",
               markersize=10, label="Marquis2019"),
    plt.Line2D([0], [0], marker="^", color="w", markerfacecolor="gray",
               markersize=10, label="OKane2022"),
    plt.Line2D([0], [0], marker="v", color="w", markerfacecolor="gray",
               markersize=10, label="ORegan2022"),
]
ax.legend(handles=m_handles, loc="upper right", fontsize=8,
          title="set marker (within family)", framealpha=0.9)

plt.tight_layout()
out_b1 = FIG_DIR / "B1_scatter_overview.png"
fig.savefig(out_b1, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [wrote] {out_b1}")


# ==========================================================================
# Figure B2 — heatmap: A_delta_t_mean_s
# ==========================================================================
print("Generating B2 heatmap (A_mean) ...")
DCAC_CONDS = [c for c in CONDITION_ORDER if c not in ("DC_0p2C", "DC_0p5C")]

# Build matrix: rows = sets (in SET_ORDER), cols = DCAC conditions
mat = np.full((len(SET_ORDER), len(DCAC_CONDS)), np.nan)
for i, s in enumerate(SET_ORDER):
    for j, c in enumerate(DCAC_CONDS):
        sub = dcac[(dcac["param_set"] == s) & (dcac["condition"] == c)]
        if len(sub) == 1:
            mat[i, j] = sub["A_delta_t_mean_s"].iloc[0]

fig, ax = plt.subplots(figsize=(7, 6))
# Diverging colormap centered at 0
vmax_abs = max(abs(np.nanmin(mat)), abs(np.nanmax(mat)))
im = ax.imshow(mat, cmap="RdBu", vmin=-vmax_abs, vmax=+vmax_abs, aspect="auto")
ax.set_xticks(range(len(DCAC_CONDS)))
ax.set_xticklabels([CONDITION_LABEL[c] for c in DCAC_CONDS], fontsize=9)
ax.set_yticks(range(len(SET_ORDER)))
ax.set_yticklabels([f"{s}\n[{CHEM_FAMILY[s]}]" for s in SET_ORDER], fontsize=8)

# Annotate cells (or "infeasible" for missing)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        if np.isnan(v):
            txt = "infeas."
            ax.text(j, i, txt, ha="center", va="center", fontsize=7,
                    color="black", fontstyle="italic")
        else:
            txt = f"{v:+.0f}"
            color = "white" if abs(v) > 0.6 * vmax_abs else "black"
            ax.text(j, i, txt, ha="center", va="center",
                    fontsize=8, color=color)

cbar = plt.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label(r"$A_{\Delta t,\mathrm{mean}}$  [s]", fontsize=10)
ax.set_title("B2: Window-averaged Δt(Q) heatmap\n(>0 = DCAC faster than DC ref)",
             fontsize=11)

plt.tight_layout()
out_b2 = FIG_DIR / "B2_heatmap_Amean.png"
fig.savefig(out_b2, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [wrote] {out_b2}")


# ==========================================================================
# Figure B3 — heatmap: sign_class_rel
# ==========================================================================
print("Generating B3 heatmap (sign_class_rel) ...")
SIGN_LABEL_TO_INT = {
    "near_zero": 0,
    "negative_only": -1,
    "positive_only": +1,
    "mixed": +2,
    "infeasible": -2,
}
SIGN_INT_TO_LABEL = {v: k for k, v in SIGN_LABEL_TO_INT.items()}

mat_sign = np.full((len(SET_ORDER), len(DCAC_CONDS)), -2)  # default = infeasible
for i, s in enumerate(SET_ORDER):
    for j, c in enumerate(DCAC_CONDS):
        sub = dcac[(dcac["param_set"] == s) & (dcac["condition"] == c)]
        if len(sub) == 1:
            cls = sub["sign_class_rel"].iloc[0]
            mat_sign[i, j] = SIGN_LABEL_TO_INT.get(cls, -2)

# Discrete colormap
sign_colors = {
    -2: "#cccccc",  # infeasible (gray)
    -1: "#377eb8",  # negative_only (blue)
     0: "#f0f0f0",  # near_zero (very light)
    +1: "#e41a1c",  # positive_only (red)
    +2: "#984ea3",  # mixed (purple)
}
cmap = ListedColormap([sign_colors[k] for k in [-2, -1, 0, 1, 2]])
norm = BoundaryNorm([-2.5, -1.5, -0.5, 0.5, 1.5, 2.5], cmap.N)

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(mat_sign, cmap=cmap, norm=norm, aspect="auto")
ax.set_xticks(range(len(DCAC_CONDS)))
ax.set_xticklabels([CONDITION_LABEL[c] for c in DCAC_CONDS], fontsize=9)
ax.set_yticks(range(len(SET_ORDER)))
ax.set_yticklabels([f"{s}\n[{CHEM_FAMILY[s]}]" for s in SET_ORDER], fontsize=8)

for i in range(mat_sign.shape[0]):
    for j in range(mat_sign.shape[1]):
        v = mat_sign[i, j]
        label_short = {
            -2: "—",
            -1: "neg",
             0: "≈0",
            +1: "pos",
            +2: "mix",
        }[v]
        ax.text(j, i, label_short, ha="center", va="center", fontsize=9,
                color="black")

# Legend
sign_handles = [
    mpatches.Patch(color=sign_colors[-1], label="negative_only"),
    mpatches.Patch(color=sign_colors[0],  label="near_zero"),
    mpatches.Patch(color=sign_colors[+1], label="positive_only (none observed)"),
    mpatches.Patch(color=sign_colors[+2], label="mixed (none observed)"),
    mpatches.Patch(color=sign_colors[-2], label="infeasible_Vmin_event"),
]
ax.legend(handles=sign_handles, loc="upper left", bbox_to_anchor=(1.02, 1.0),
          fontsize=8, framealpha=0.9)
ax.set_title("B3: Sign class (rel; 0.5% of DC ref t_charge threshold)",
             fontsize=11)

plt.tight_layout()
out_b3 = FIG_DIR / "B3_heatmap_signclass.png"
fig.savefig(out_b3, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [wrote] {out_b3}")


# ==========================================================================
# Figure B4 — Δt(Q) curves overlaid at DC0p2_AC0p5_10tau
# ==========================================================================
print("Generating B4 curve overlay (DC0p2_AC0p5_10tau) ...")
COND_B4 = "DC0p2_AC0p5_10tau"

fig, ax = plt.subplots(figsize=(9, 6))

for s in SET_ORDER:
    sub = df_curves[(df_curves["param_set"] == s) &
                    (df_curves["condition"] == COND_B4)].copy()
    if len(sub) == 0:
        continue
    sub = sub.sort_values("Q_Ah")
    fam = CHEM_FAMILY[s]
    color = FAMILY_COLOR[fam]
    # Normalize x to SoC for cross-set comparison
    Q_nom = sub["Q_nom_Ah"].iloc[0]
    soc = sub["Q_Ah"] / Q_nom * 100  # additional SoC % above set_initial_state(0.05) start
    ax.plot(soc, sub["dtQ_s"], color=color, linewidth=1.6,
            marker=SET_MARKER[s], markersize=4, markevery=10,
            label=f"{s} [{fam}]", alpha=0.85)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Charge throughput / Q_nom  [%]   (start ≈ 5% SoC)", fontsize=11)
ax.set_ylabel("Δt(Q)  [s]   (>0 = DCAC faster)", fontsize=11)
ax.set_title(f"B4: Δt(Q) curves at {CONDITION_LABEL[COND_B4].replace(chr(10),' ')}\n"
             f"all 8 parameter sets admissible at this condition", fontsize=11)
ax.legend(loc="lower right", fontsize=8, framealpha=0.9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
out_b4 = FIG_DIR / "B4_curves_AC0p5_10tau.png"
fig.savefig(out_b4, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [wrote] {out_b4}")


# ==========================================================================
# Figure B5 — feasibility envelope (Cell 2 CSV; includes infeasible)
# ==========================================================================
print("Generating B5 feasibility envelope ...")
VERDICT_TO_INT = {
    "feasible_clean":        2,
    "feasible_transient":    1,
    "boundary_confounded":   0,
    "infeasible_Vmin_event": -1,
    "infeasible_no_Vmax":    -2,
    "solve_error":           -3,
}
INT_TO_VERDICT = {v: k for k, v in VERDICT_TO_INT.items()}
VERDICT_COLOR = {
    "feasible_clean":        "#1a9850",
    "feasible_transient":    "#a6d96a",
    "boundary_confounded":   "#fee08b",
    "infeasible_Vmin_event": "#d73027",
    "infeasible_no_Vmax":    "#7f3b08",
    "solve_error":           "#000000",
}

mat_feas = np.full((len(SET_ORDER), len(CONDITION_ORDER)), np.nan)
verdicts = np.empty((len(SET_ORDER), len(CONDITION_ORDER)), dtype=object)
for i, s in enumerate(SET_ORDER):
    for j, c in enumerate(CONDITION_ORDER):
        sub = df_feas[(df_feas["param_set"] == s) & (df_feas["condition"] == c)]
        if len(sub) == 1:
            v = sub["feasibility_verdict"].iloc[0]
            mat_feas[i, j] = VERDICT_TO_INT.get(v, np.nan)
            verdicts[i, j] = v

# CRITICAL: cmap colors and BoundaryNorm boundaries MUST share the same
# integer-sorted ordering. Build present_verdicts FROM present (sorted ints)
# rather than from VERDICT_TO_INT dict-insertion order.
present_ints = sorted(set(int(v) for v in np.unique(mat_feas) if not np.isnan(v)))
present_verdicts = [INT_TO_VERDICT[i] for i in present_ints]
cmap_b5 = ListedColormap([VERDICT_COLOR[v] for v in present_verdicts])
boundaries = [present_ints[0] - 0.5] + [v + 0.5 for v in present_ints]
norm_b5 = BoundaryNorm(boundaries, cmap_b5.N)

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(mat_feas, cmap=cmap_b5, norm=norm_b5, aspect="auto")
ax.set_xticks(range(len(CONDITION_ORDER)))
ax.set_xticklabels([CONDITION_LABEL[c] for c in CONDITION_ORDER], fontsize=9)
ax.set_yticks(range(len(SET_ORDER)))
ax.set_yticklabels([f"{s}\n[{CHEM_FAMILY[s]}]" for s in SET_ORDER], fontsize=8)

# Cell text — color depends on actual verdict, not assumed cmap order
DARK_VERDICTS = {"infeasible_Vmin_event", "infeasible_no_Vmax", "solve_error"}
SHORT = {
    "feasible_clean":        "✓",
    "feasible_transient":    "~",
    "boundary_confounded":   "?",
    "infeasible_Vmin_event": "✗\nVmin",
    "infeasible_no_Vmax":    "✗\nnoVmax",
    "solve_error":           "ERR",
}
for i in range(verdicts.shape[0]):
    for j in range(verdicts.shape[1]):
        v = verdicts[i, j]
        if v is None:
            continue
        ax.text(j, i, SHORT[v], ha="center", va="center", fontsize=8,
                color="white" if v in DARK_VERDICTS else "black")

handles_b5 = [mpatches.Patch(color=VERDICT_COLOR[v], label=v)
              for v in present_verdicts]
ax.legend(handles=handles_b5, loc="upper left", bbox_to_anchor=(1.02, 1.0),
          fontsize=8, framealpha=0.9)
ax.set_title("B5: Cell 2 feasibility envelope (all 48 simulations)\n"
             "infeasible_Vmin_event clusters at DC=0.2C / AC=1.0C / 10τ on 5 Ah sets",
             fontsize=10)

plt.tight_layout()
out_b5 = FIG_DIR / "B5_feasibility_envelope.png"
fig.savefig(out_b5, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"  [wrote] {out_b5}")


# ==========================================================================
# Summary
# ==========================================================================
print(f"\n{'='*60}")
print(f"5 figures written to: {FIG_DIR}")
print(f"{'='*60}")
for f in sorted(FIG_DIR.glob("B*.png")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

=== Cell 3B — Day 16 visualization (5 figures) ===

Feasibility rows: 48
Δt(Q) summary rows: 36  (28 DCAC + 8 sanity)
Δt(Q) curve points: 3600

Generating B1 scatter overview ...
  [wrote] /Users/louislu/pybamm-dcac-superimposed/figures/day16/B1_scatter_overview.png
Generating B2 heatmap (A_mean) ...
  [wrote] /Users/louislu/pybamm-dcac-superimposed/figures/day16/B2_heatmap_Amean.png
Generating B3 heatmap (sign_class_rel) ...
  [wrote] /Users/louislu/pybamm-dcac-superimposed/figures/day16/B3_heatmap_signclass.png
Generating B4 curve overlay (DC0p2_AC0p5_10tau) ...
  [wrote] /Users/louislu/pybamm-dcac-superimposed/figures/day16/B4_curves_AC0p5_10tau.png
Generating B5 feasibility envelope ...
  [wrote] /Users/louislu/pybamm-dcac-superimposed/figures/day16/B5_feasibility_envelope.png

5 figures written to: /Users/louislu/pybamm-dcac-superimposed/figures/day16
  B1_scatter_overview.png  (83 KB)
  B2_heatmap_Amean.png  (79 KB)
  B3_heatmap_signclass.png  (67 KB)
  B4_curves_AC0p5_10tau.png 